In [1]:
%cd ..

/media/kirti/Dev/GenAI/TransformerFromScratch


## Data Creation

In [2]:
data_path = "data/kirti.txt"
with open(data_path,"r") as file:
    data = file.read()

len(data)

2237

In [3]:
unique_character = sorted(list(set(data)))

In [4]:
vocab_size = len(unique_character)

In [5]:
int_to_string = {k:v for k,v in enumerate(unique_character)}
string_to_int = {v:k for k,v in enumerate(unique_character)}

In [6]:
string_to_int

{'\n': 0,
 ' ': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '-': 6,
 '.': 7,
 '/': 8,
 ':': 9,
 '?': 10,
 'A': 11,
 'C': 12,
 'D': 13,
 'E': 14,
 'G': 15,
 'I': 16,
 'K': 17,
 'L': 18,
 'M': 19,
 'N': 20,
 'P': 21,
 'Q': 22,
 'T': 23,
 'W': 24,
 'Y': 25,
 'a': 26,
 'b': 27,
 'c': 28,
 'd': 29,
 'e': 30,
 'f': 31,
 'g': 32,
 'h': 33,
 'i': 34,
 'j': 35,
 'k': 36,
 'l': 37,
 'm': 38,
 'n': 39,
 'o': 40,
 'p': 41,
 'r': 42,
 's': 43,
 't': 44,
 'u': 45,
 'v': 46,
 'w': 47,
 'y': 48,
 'z': 49}

In [7]:
def encode(s):
    ans = []
    for char in s:
        ans.append(string_to_int[char])
    return ans

def decode(l):
    ans = ""
    for token in l:
        ans+=int_to_string[token]
    return ans


In [8]:
encode("Kirti")

[17, 34, 42, 44, 34]

In [9]:
decode([17, 34, 42, 44, 34])

'Kirti'

In [10]:
import torch


In [11]:
encoded_data = torch.tensor(encode(data),dtype=torch.long)

In [12]:
len(encoded_data)*90//100

2013

In [13]:
encoded_data.dtype

torch.int64

In [14]:
split_idx = data.rfind("\n\n", 0, int(len(data) * 0.9))

In [15]:
train_data = encoded_data[:split_idx]
val_data = encoded_data[split_idx:]

In [16]:
chunk = train_data[:9]



In [17]:
x = chunk[:-1]
y = chunk[1:]

In [18]:
x,y

(tensor([17, 34, 42, 44, 34,  1, 21, 40]),
 tensor([34, 42, 44, 34,  1, 21, 40, 32]))

In [19]:
torch.randint(0,len(train_data)-8,(4,1))

tensor([[1356],
        [ 265],
        [ 701],
        [ 580]])

In [20]:
batch_size = 4
block_size = 8
ix = torch.randint(0,len(train_data)-block_size,(batch_size,))

In [21]:
x = torch.stack([train_data[i:i+block_size] for i in ix])
y = torch.stack([train_data[i+1:i+block_size+1] for i in ix])

In [22]:
x,y

(tensor([[42, 30, 30,  1, 44, 34, 38, 30],
         [34, 42, 44, 34,  1, 21, 40, 32],
         [37, 29, 34, 39, 32,  1, 26,  1],
         [39, 32,  1, 26, 39, 29,  1, 47]]),
 tensor([[30, 30,  1, 44, 34, 38, 30,  5],
         [42, 44, 34,  1, 21, 40, 32, 42],
         [29, 34, 39, 32,  1, 26,  1, 15],
         [32,  1, 26, 39, 29,  1, 47, 40]]))

In [23]:
x.shape,y.shape

(torch.Size([4, 8]), torch.Size([4, 8]))

In [24]:
def get_batch(split_type):
    batch_size = 4
    block_size = 8
    data = train_data if split_type=="train" else val_data
    ix = torch.randint(0,len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y
                

In [25]:
get_batch("train")

(tensor([[40, 29, 30, 37,  1, 31, 42, 40],
         [11,  1, 29, 30, 32, 42, 30, 30],
         [ 9,  1, 17, 34, 42, 44, 34,  1],
         [42,  1, 47, 40, 42, 36, 34, 39]]),
 tensor([[29, 30, 37,  1, 31, 42, 40, 38],
         [ 1, 29, 30, 32, 42, 30, 30,  1],
         [ 1, 17, 34, 42, 44, 34,  1, 34],
         [ 1, 47, 40, 42, 36, 34, 39, 32]]))

In [26]:
get_batch("val")

(tensor([[26, 31, 44, 43,  7,  0, 17, 34],
         [45, 43, 34, 28,  5,  1, 26, 39],
         [ 1, 26, 39, 29,  1, 39, 40, 47],
         [ 1, 26, 39, 29,  1, 28, 42, 26]]),
 tensor([[31, 44, 43,  7,  0, 17, 34, 42],
         [43, 34, 28,  5,  1, 26, 39, 29],
         [26, 39, 29,  1, 39, 40, 47,  1],
         [26, 39, 29,  1, 28, 42, 26, 31]]))

## Training

In [27]:
from torch import nn
from torch.nn import functional

In [28]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size:int
    d_model:int
    max_seq_len: int # context length
    num_heads:int
    num_layers:int
    dropout:float

In [31]:
model_config = ModelConfig(
    vocab_size=50,
    d_model=64,
    max_seq_len=25,
    num_heads=8,
    num_layers=4,
    dropout=0.2
)

In [32]:
class EmbeddingLayer(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        """
        Always remember that embedding layer is not actually a neural laer with any activation function ,
        its just an 2d matrix lookup the matrix size is (vocab_size,d_model) but but but the d_model here is lets one array
         so that array is already have learnable parameter means in our case that 32 dim arrays have the values from the training 
         for e.g:
         token_embeding[1] will be the embeding of token id 1 which is 32 dim embedding array
        """
        self.token_embeddings=nn.Embedding(model_config.vocab_size,model_config.d_model)
        self.pos_embeddings = nn.Embedding(model_config.max_seq_len,model_config.d_model)

    def forward(self,token_ids):
        pos_ids = torch.arange(token_ids.shape[1],device=token_ids.device)
        embedding = self.token_embeddings(token_ids) + self.pos_embeddings(pos_ids)
        return embedding


In [33]:
class MultiHeadAttention(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        self.config = model_config
        # initialize q,k,v
        self.w_q = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_k = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_v = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

        self.register_buffer(
                            "casual_mask", 
                            torch.tril(
                                torch.ones(
                                    self.config.max_seq_len, 
                                    self.config.max_seq_len
                                    )
                                )
                        )
        
        self.head_dim = self.config.d_model//self.config.num_heads
        

        self.w_o = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

    def forward(self,x):
        # here x is the embeding and not raw x  and b is batch size, t is the current sequence length and d_model is hidden dimension
        # shape of x -> (b,t,d_model)
        q = self.w_q(x) # (b,t,d_model)
        k = self.w_k(x) # (b,t,d_model)
        v = self.w_v(x) # (b,t,d_model)

        q = torch.reshape(q,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        k = torch.reshape(k,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        v = torch.reshape(v,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)


        q = q.transpose(1,2) # (b,num_head,t,head_dim)
        k = k.transpose(1,2) # (b,num_head,t,head_dim)
        v = v.transpose(1,2) # (b,num_head,t,head_dim)

        scores = q @ k.transpose(-2,-1) / (self.head_dim ** 0.5) # (b,num_head,t,t)
        mask = self.casual_mask[:x.shape[1],:x.shape[1]] # (t,t)
        masked_score = scores.masked_fill(mask==0,float("-inf")) # (b,num_head,t,t)
        attention_weights = torch.softmax(masked_score,dim = -1) # (b,num_head,t,t)
        context_vector = attention_weights @ v # (b,num_head,t,head_dim)

        context_vector = context_vector.transpose(1,2) # (b,t,num_head,head_dim) 
        context_vector = context_vector.contiguous().view(x.shape) # (b,t,d_model)
        return self.w_o(context_vector) #(b,t,d_model) which is same as the input embedding 


In [34]:
class FeedForward(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()
        self.config = model_config
        self.model = nn.Sequential(
            nn.Linear(self.config.d_model,4*self.config.d_model),
            nn.GELU(),
            nn.Linear(4*self.config.d_model,self.config.d_model)
        )

    def forward(self,x):
        return self.model(x)

In [35]:
class DecoderBlock(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        self.config = model_config
        self.attention_layer = MultiHeadAttention(model_config)
        self.ffn = FeedForward(model_config)
        self.ln_1 =  nn.LayerNorm(model_config.d_model) # before attention layer
        self.ln_2 =  nn.LayerNorm(model_config.d_model) # before feed forward layer

    def forward(self,x):
        x =  x+self.attention_layer(self.ln_1(x))
        x = x+ self.ffn(self.ln_2(x))
        return x



In [36]:
class Transformer(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()
        self.config = model_config
        self.embedding_layer = EmbeddingLayer(model_config)
        self.drop = nn.Dropout(model_config.dropout)
        self.block = nn.ModuleList([DecoderBlock(model_config) for _ in range(model_config.num_layers)])
        self.ln_f = nn.LayerNorm(model_config.d_model)
        self.lm_head = nn.Linear(model_config.d_model,model_config.vocab_size,bias=False)
        self.lm_head.weight = self.embedding_layer.token_embeddings.weight

        self.apply(self._init_weights)

    def _init_weights(self,module):
        if isinstance(module, nn.Linear):
            module.weight = torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
            if module.bias is not None:
                module.bias = torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            module.weight = nn.init.normal_(module.weight,mean=0,std=0.02)

        
    def forward(self,x,targets=None):
        embedding = self.embedding_layer(x)
        out = self.drop(embedding)
        for i in range(self.config.num_layers):
            out = self.block[i](out)

        out_norm = self.ln_f(out)
        logits = self.lm_head(out_norm)
        loss = None
        if targets is not None:
            loss = functional.cross_entropy(
                logits.view(-1,self.config.vocab_size),
                targets.view(-1))
        return logits,loss

    def generate(self,idx,max_new_tokens,temperature=1.0,top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:,-self.config.max_seq_len:] #index conditional
            logits,_ = self(idx_cond)
            curr_id=0
            last_token = logits[:,-1,:]
            if temperature == 0.0: 
                curr_id = torch.argmax(last_token,dim=-1,keepdim=True)
            elif temperature > 0:
                last_token = last_token/temperature 
                if top_k is not None:
                    cutoff,_ = torch.topk(last_token,k=top_k)
                    last_token = last_token.masked_fill(last_token<cutoff[:,[-1]],float("-inf"))
                probs = functional.softmax(last_token,dim=-1)
                curr_id = torch.multinomial(probs,num_samples=1)
            idx = torch.cat((idx, curr_id), dim=1)

        return idx

In [37]:
def get_batch(split_type,batch_size,block_size):
    data = train_data if split_type=="train" else val_data
    ix = torch.randint(0,len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y
                

In [38]:
model = Transformer(model_config)

In [39]:
model

Transformer(
  (embedding_layer): EmbeddingLayer(
    (token_embeddings): Embedding(50, 64)
    (pos_embeddings): Embedding(25, 64)
  )
  (drop): Dropout(p=0.2, inplace=False)
  (block): ModuleList(
    (0-3): 4 x DecoderBlock(
      (attention_layer): MultiHeadAttention(
        (w_q): Linear(in_features=64, out_features=64, bias=False)
        (w_k): Linear(in_features=64, out_features=64, bias=False)
        (w_v): Linear(in_features=64, out_features=64, bias=False)
        (w_o): Linear(in_features=64, out_features=64, bias=False)
      )
      (ffn): FeedForward(
        (model): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
      )
      (ln_1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln_2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (ln_f): LayerNorm((64,), eps=1e-05, 

In [40]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 203,840


In [41]:
xb,yb = get_batch("train",4,model_config.max_seq_len)

In [42]:
logits,loss = model(xb,yb)

In [43]:
loss.item(),logits.shape

(3.9231808185577393, torch.Size([4, 25, 50]))

In [44]:
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)

In [45]:
optimizer.zero_grad(set_to_none=True)

In [46]:
loss.backward()

In [47]:
optimizer.step()

In [48]:
_,new_loss = model(xb,yb)
print("Before:", loss.item(), "After:", new_loss.item())

Before: 3.9231808185577393 After: 3.6769814491271973


In [49]:
@torch.no_grad()
def calculate_loss(eval_iters=20,batch_size=4):
    out = {}
    model.eval()
    train_loss,val_loss=[],[]
    for _ in range(eval_iters):
        x_train,y_train = get_batch("train",batch_size,model_config.max_seq_len)
        _,loss = model(x_train,y_train)
        train_loss.append(loss.item())

        x_val,y_val = get_batch("val",batch_size,model_config.max_seq_len)
        _,loss = model(x_val,y_val)
        val_loss.append(loss.item())

    out = {"train_loss": sum(train_loss)/eval_iters,
           "val_loss":sum(val_loss)/eval_iters}
    model.train()
    return out


In [50]:
max_iters = 2500    # total training steps
eval_interval = 250 # evaluate every 50 steps
batch_size = 8
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)

In [51]:
for iter in range(max_iters):
    xb,yb = get_batch("train",batch_size,model_config.max_seq_len)
    _,loss = model(xb,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if iter % eval_interval == 0:
        out = calculate_loss(batch_size=batch_size)
        print(out)

{'train_loss': 3.658767580986023, 'val_loss': 3.659927523136139}
{'train_loss': 1.8626880526542664, 'val_loss': 2.1273779571056366}
{'train_loss': 1.2487686634063722, 'val_loss': 1.7530967891216278}
{'train_loss': 0.8604145437479019, 'val_loss': 1.4862887263298035}
{'train_loss': 0.6034065246582031, 'val_loss': 1.683024799823761}
{'train_loss': 0.448905947804451, 'val_loss': 1.6064546525478363}
{'train_loss': 0.37747129052877426, 'val_loss': 1.4648556232452392}
{'train_loss': 0.3268841728568077, 'val_loss': 1.6941033363342286}
{'train_loss': 0.3274313256144524, 'val_loss': 1.714094376564026}
{'train_loss': 0.2967421695590019, 'val_loss': 1.6953702867031097}


In [52]:
prompt = torch.tensor([encode("Q: Who ")], dtype=torch.long)
out = model.generate(prompt, max_new_tokens=60,temperature=0.0,top_k=5)

In [53]:
decode(out[0].tolist())

'Q: Who is Kirti Pogra?\nA: Kirti Pogra is an AI/ML Engineeer who com'

In [54]:
prompt = torch.tensor([encode("Q: Who is Kirti?\nA:")], dtype=torch.long)

In [55]:
out = model.generate(prompt, max_new_tokens=60,temperature=0.1)
decode(out[0].tolist())

'Q: Who is Kirti?\nA: Kirti wants to build a staching songs, and working as an AI'

In [56]:
prompt = torch.tensor([encode("Kirti is ")], dtype=torch.long)
out = model.generate(prompt, max_new_tokens=60,temperature=0.1)
decode(out[0].tolist())

'Kirti is currrently building a GPT-style generative Toransformer and '

In [57]:
prompt = torch.tensor([encode("Q: What degree ")], dtype=torch.long)
out = model.generate(prompt, max_new_tokens=60,temperature=0.7)
decode(out[0].tolist())

'Q: What degree does Kirti Pogra from?\nA: Kirti Pogra is an AI/ML Ending are'

In [58]:
checkpoint = {
    'model_state': model.state_dict(),
    'config': model_config,
    'stoi': string_to_int,
    'itos': int_to_string
}

In [59]:
torch.save(checkpoint, 'transformer_model.pt')

In [60]:
checkpoint = torch.load("transformer_model.pt",weights_only=False)

In [61]:
loaded_model = Transformer(checkpoint['config'])

In [62]:
loaded_model.load_state_dict(checkpoint["model_state"])

<All keys matched successfully>

In [63]:
loaded_model.eval()

Transformer(
  (embedding_layer): EmbeddingLayer(
    (token_embeddings): Embedding(50, 64)
    (pos_embeddings): Embedding(25, 64)
  )
  (drop): Dropout(p=0.2, inplace=False)
  (block): ModuleList(
    (0-3): 4 x DecoderBlock(
      (attention_layer): MultiHeadAttention(
        (w_q): Linear(in_features=64, out_features=64, bias=False)
        (w_k): Linear(in_features=64, out_features=64, bias=False)
        (w_v): Linear(in_features=64, out_features=64, bias=False)
        (w_o): Linear(in_features=64, out_features=64, bias=False)
      )
      (ffn): FeedForward(
        (model): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
      )
      (ln_1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln_2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (ln_f): LayerNorm((64,), eps=1e-05, 